# 05 - Real vs Synthetic (multi-model)

Reads exclusively from `ai_narratives_original` via `pain_narratives.analysis.revision_data_layer`. Iterates over every model present in the schema (GPT-5, DeepSeek-R1, Claude Sonnet 4.5).

**Outputs** under `notebooks/outputs/`:
- `05_real_vs_synthetic_all.csv` - per-model per-questionnaire MAE/RMSE/Pearson/Spearman
- `05_cronbach_alpha.csv` - per-model Cronbach alpha (real + synthetic mean+/-SD across runs)
- `05_llm_consistency.csv` - run-level consistency SD per participant per model
- `05_dim_vs_real_quest.csv` - synthetic dimension scores correlated with real patient questionnaires (T10e analog)


## 1. Setup

In [1]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / 'src'))
from pain_narratives.analysis import revision_data_layer as dl

OUT = Path.cwd() / 'outputs'
OUT.mkdir(exist_ok=True, parents=True)

MODELS = dl.available_models()
print(f'Models in scope: {MODELS}')


Models in scope: ['claude-sonnet-4-5-thinking', 'deepseek-r1', 'gpt-5']


## 2. Load real data + synthetic data per model

In [2]:
real = dl.load_real_from_schema()
print(f'Real data: {len(real)} narratives')
print(f'  PCS  mean: {real["pcs_total"].mean():.2f} +/- {real["pcs_total"].std():.2f}')
print(f'  BPI  mean: {real["bpi_total_mean"].mean():.2f} +/- {real["bpi_total_mean"].std():.2f}')
print(f'  TSK  mean: {real["tsk_total"].mean():.2f} +/- {real["tsk_total"].std():.2f}')

synth = {m: dl.load_synth_from_schema(m) for m in MODELS}
for m, df in synth.items():
    n_runs = df['run_number'].nunique()
    print(f'  {m}: {len(df)} rows, {n_runs} runs')


Real data: 152 narratives
  PCS  mean: 31.44 +/- 11.71
  BPI  mean: 6.90 +/- 1.68
  TSK  mean: 28.30 +/- 6.26


  claude-sonnet-4-5-thinking: 456 rows, 3 runs
  deepseek-r1: 456 rows, 3 runs
  gpt-5: 456 rows, 3 runs


## 3. Helpers

In [3]:
QUESTIONNAIRES = ['pcs_total', 'bpi_total_mean', 'bpi_interference', 'bpi_intensity', 'tsk_total']
# Map output label -> (real column, synth column) since real uses *_mean and synth uses *_avg.
Q_COLS = {
    'pcs_total': ('pcs_total', 'pcs_total'),
    'bpi_total_mean': ('bpi_total_mean', 'bpi_total_mean'),
    'bpi_interference': ('bpi_interference_mean', 'bpi_interference_avg'),
    'bpi_intensity': ('bpi_intensity_mean', 'bpi_intensity_avg'),
    'tsk_total': ('tsk_total', 'tsk_total'),
}

PCS_ITEMS = [f'pcs_{i:02d}' for i in range(1, 14)]
BPI_INTERFERENCE_ITEMS = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq15', 'bpiq16', 'bpiq17']
BPI_INTENSITY_ITEMS = ['bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
TSK_ITEMS = [f'tsk_{i:02d}' for i in range(1, 12)]

def cronbach_alpha(df, items):
    df = df[items].apply(pd.to_numeric, errors='coerce').dropna(how='all').dropna(how='any')
    if len(df) < 3:
        return np.nan
    k = len(items)
    item_var = df.var(ddof=1).sum()
    total_var = df.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return np.nan
    return float(k / (k - 1) * (1 - item_var / total_var))

def synth_per_narrative(df_model, columns):
    '''Mean across runs for each (participant, column).'''
    cols = [c for c in columns if c in df_model.columns]
    return df_model.groupby('participant_id', as_index=False)[cols].mean()


## 4. Per-model MAE / RMSE / Pearson / Spearman (Table T3)

Per-narrative mean across runs vs real patient questionnaire totals.

In [4]:
def agreement(real_v, synth_v):
    m = ~(real_v.isna() | synth_v.isna())
    r = real_v[m].to_numpy(dtype=float); s = synth_v[m].to_numpy(dtype=float)
    n = len(r)
    if n < 3:
        return dict(n=n, mae=np.nan, rmse=np.nan, pearson_r=np.nan, spearman=np.nan)
    pr, _ = stats.pearsonr(r, s)
    sp, _ = stats.spearmanr(r, s)
    return dict(
        n=n,
        mae=float(np.mean(np.abs(r - s))),
        rmse=float(np.sqrt(np.mean((r - s) ** 2))),
        pearson_r=float(pr), spearman=float(sp),
    )

rows = []
for m in MODELS:
    agg = synth_per_narrative(synth[m], [v[1] for v in Q_COLS.values()])
    merged = real.merge(agg, on='participant_id', suffixes=('_real', '_synth'))
    for label, (real_col, synth_col) in Q_COLS.items():
        synth_actual = synth_col if synth_col not in (real_col,) else synth_col + '_synth'
        real_actual = real_col if real_col not in (synth_col,) else real_col + '_real'
        if real_col == synth_col:
            real_actual, synth_actual = real_col + '_real', synth_col + '_synth'
        rows.append(dict(model=m, questionnaire=label,
                         **agreement(merged[real_actual], merged[synth_actual])))
df_agree = pd.DataFrame(rows)
df_agree.to_csv(OUT / '05_real_vs_synthetic_all.csv', index=False)
print(df_agree.round(3).to_string(index=False))


                     model    questionnaire   n    mae   rmse  pearson_r  spearman
claude-sonnet-4-5-thinking        pcs_total 152 10.752 13.744      0.470     0.450
claude-sonnet-4-5-thinking   bpi_total_mean 152  0.967  1.241      0.726     0.734
claude-sonnet-4-5-thinking bpi_interference 152  1.207  1.639      0.611     0.666
claude-sonnet-4-5-thinking    bpi_intensity 152  1.210  1.494      0.710     0.664
claude-sonnet-4-5-thinking        tsk_total 152  7.114  8.785      0.368     0.321
               deepseek-r1        pcs_total 152 10.664 13.570      0.474     0.435
               deepseek-r1   bpi_total_mean 152  1.086  1.421      0.668     0.677
               deepseek-r1 bpi_interference 152  1.347  1.818      0.552     0.611
               deepseek-r1    bpi_intensity 152  1.191  1.559      0.650     0.643
               deepseek-r1        tsk_total 152  9.121 11.223      0.312     0.281
                     gpt-5        pcs_total 152 10.963 14.029      0.444     0.419
    

## 5. Cronbach alpha per model (Table T5)

Real data: single alpha per questionnaire.
Synthetic: alpha computed per run, then mean +/- SD across runs.

In [5]:
alpha_rows = []
# Real once
alpha_rows.append(dict(source='real', model='-',
                       pcs_alpha=cronbach_alpha(real, PCS_ITEMS),
                       bpi_interference_alpha=cronbach_alpha(real, BPI_INTERFERENCE_ITEMS),
                       bpi_intensity_alpha=cronbach_alpha(real, BPI_INTENSITY_ITEMS),
                       tsk_alpha=cronbach_alpha(real, TSK_ITEMS),
                       n_runs=1))
# Per model per run, then aggregate
for m in MODELS:
    df_m = synth[m].copy()
    # TSK items are stored as strings; coerce.
    for c in TSK_ITEMS:
        if c in df_m.columns:
            df_m[c] = pd.to_numeric(df_m[c], errors='coerce')
    runs = sorted(df_m['run_number'].dropna().unique())
    per_run = []
    for r in runs:
        sub = df_m[df_m['run_number'] == r]
        per_run.append(dict(
            run=r,
            pcs_alpha=cronbach_alpha(sub, PCS_ITEMS),
            bpi_interference_alpha=cronbach_alpha(sub, BPI_INTERFERENCE_ITEMS),
            bpi_intensity_alpha=cronbach_alpha(sub, BPI_INTENSITY_ITEMS),
            tsk_alpha=cronbach_alpha(sub, TSK_ITEMS),
        ))
    df_runs = pd.DataFrame(per_run)
    alpha_rows.append(dict(
        source='synthetic_mean', model=m,
        pcs_alpha=df_runs['pcs_alpha'].mean(),
        bpi_interference_alpha=df_runs['bpi_interference_alpha'].mean(),
        bpi_intensity_alpha=df_runs['bpi_intensity_alpha'].mean(),
        tsk_alpha=df_runs['tsk_alpha'].mean(),
        n_runs=len(runs),
    ))
    alpha_rows.append(dict(
        source='synthetic_sd', model=m,
        pcs_alpha=df_runs['pcs_alpha'].std(),
        bpi_interference_alpha=df_runs['bpi_interference_alpha'].std(),
        bpi_intensity_alpha=df_runs['bpi_intensity_alpha'].std(),
        tsk_alpha=df_runs['tsk_alpha'].std(),
        n_runs=len(runs),
    ))
df_alpha = pd.DataFrame(alpha_rows)
df_alpha.to_csv(OUT / '05_cronbach_alpha.csv', index=False)
print(df_alpha.round(3).to_string(index=False))


        source                      model  pcs_alpha  bpi_interference_alpha  bpi_intensity_alpha  tsk_alpha  n_runs
          real                          -      0.928                   0.831                0.938      0.846       1
synthetic_mean claude-sonnet-4-5-thinking      0.986                   0.963                0.974      0.953       3
  synthetic_sd claude-sonnet-4-5-thinking      0.000                   0.001                0.001      0.001       3
synthetic_mean                deepseek-r1      0.972                   0.923                0.952      0.956       3
  synthetic_sd                deepseek-r1      0.001                   0.003                0.004      0.003       3
synthetic_mean                      gpt-5      0.980                   0.963                0.966      0.935       3
  synthetic_sd                      gpt-5      0.001                   0.001                0.004      0.001       3


## 6. LLM consistency across runs (Table T7)

Per participant per model: SD across runs for each total.

In [6]:
rows = []
for m in MODELS:
    df_m = synth[m]
    for col_label, col in (('pcs_total', 'pcs_total'),
                            ('bpi_total_mean', 'bpi_total_mean'),
                            ('tsk_total', 'tsk_total')):
        sd_series = df_m.groupby('participant_id')[col].std()
        rows.append(dict(
            model=m, questionnaire=col_label,
            n_runs=df_m['run_number'].nunique(),
            mean_sd_across_runs=float(sd_series.mean()),
            min_sd=float(sd_series.min()),
            max_sd=float(sd_series.max()),
        ))
df_consist = pd.DataFrame(rows)
df_consist.to_csv(OUT / '05_llm_consistency.csv', index=False)
print(df_consist.round(3).to_string(index=False))


                     model  questionnaire  n_runs  mean_sd_across_runs  min_sd  max_sd
claude-sonnet-4-5-thinking      pcs_total       3                2.039     0.0   8.505
claude-sonnet-4-5-thinking bpi_total_mean       3                0.214     0.0   0.794
claude-sonnet-4-5-thinking      tsk_total       3                1.644     0.0   6.658
               deepseek-r1      pcs_total       3                2.325     0.0   7.000
               deepseek-r1 bpi_total_mean       3                0.250     0.0   1.150
               deepseek-r1      tsk_total       3                1.767     0.0   5.859
                     gpt-5      pcs_total       3                2.223     0.0   6.658
                     gpt-5 bpi_total_mean       3                0.205     0.0   0.611
                     gpt-5      tsk_total       3                1.819     0.0   8.145


## 7. Synthetic dimension vs real questionnaire (Table T10e analog)

For each model, correlate the per-narrative mean Severidad / Discapacidad with the
real patient PCS / BPI / TSK totals. Answers: does the LLM's dimension assessment
predict the real patient's questionnaire totals?

In [7]:
DIMS = ['severidad_score', 'discapacidad_score']
REAL_TARGETS = ['pcs_total', 'bpi_total_mean',
                'bpi_interference_mean', 'bpi_intensity_mean', 'tsk_total']
rows = []
for m in MODELS:
    agg = synth_per_narrative(synth[m], DIMS)
    merged = real.merge(agg, on='participant_id')
    for d in DIMS:
        for q in REAL_TARGETS:
            mask = ~(merged[d].isna() | merged[q].isna())
            x = merged.loc[mask, d].to_numpy(dtype=float)
            y = merged.loc[mask, q].to_numpy(dtype=float)
            n = len(x)
            if n < 4:
                rho = np.nan; p = np.nan
            else:
                rho, p = stats.spearmanr(x, y)
            rows.append(dict(model=m, dimension=d, real_questionnaire=q,
                             n=n, spearman_rho=float(rho), p_value=float(p)))
df_dim = pd.DataFrame(rows)
df_dim.to_csv(OUT / '05_dim_vs_real_quest.csv', index=False)
print(df_dim.round(3).to_string(index=False))


                     model          dimension    real_questionnaire   n  spearman_rho  p_value
claude-sonnet-4-5-thinking    severidad_score             pcs_total 152         0.357    0.000
claude-sonnet-4-5-thinking    severidad_score        bpi_total_mean 152         0.603    0.000
claude-sonnet-4-5-thinking    severidad_score bpi_interference_mean 152         0.549    0.000
claude-sonnet-4-5-thinking    severidad_score    bpi_intensity_mean 152         0.520    0.000
claude-sonnet-4-5-thinking    severidad_score             tsk_total 152         0.207    0.010
claude-sonnet-4-5-thinking discapacidad_score             pcs_total 151         0.358    0.000
claude-sonnet-4-5-thinking discapacidad_score        bpi_total_mean 151         0.581    0.000
claude-sonnet-4-5-thinking discapacidad_score bpi_interference_mean 151         0.542    0.000
claude-sonnet-4-5-thinking discapacidad_score    bpi_intensity_mean 151         0.512    0.000
claude-sonnet-4-5-thinking discapacidad_score     

## 8. Done

In [8]:
print('Outputs written:')
for name in ('05_real_vs_synthetic_all.csv',
             '05_cronbach_alpha.csv',
             '05_llm_consistency.csv',
             '05_dim_vs_real_quest.csv'):
    print(f'  {OUT / name}')


Outputs written:
  /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/05_real_vs_synthetic_all.csv
  /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/05_cronbach_alpha.csv
  /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/05_llm_consistency.csv
  /Users/gferreira/Documents/my_repos/pain-narratives-app-public/notebooks/outputs/05_dim_vs_real_quest.csv
